# 03 · 环境初始化与 MAS 基础

> 完整目录见 [`README.md`](README.md) · 建议先完成 **01–02** 架构 Demo

## AutoGen 与 CrewAI — AI Engineer 训练营

---

### 本 Notebook 覆盖内容

| Part | 内容 | 难度 |
|------|------|------|
| Part 2 | AutoGen 实战（GraphFlow / SelectorGroupChat / Memory） | ⭐⭐ |
| Part 3 | CrewAI 实战（顺序 / 层级 / 工具集成） | ⭐⭐ |
| Part 4 | Agentic RAG 多 Agent（检索/生成/验证/编排） | ⭐⭐⭐ |
| Part 5 | 练习 A — 三角色流水线（入门） | ⭐ |
| Part 6 | 练习 B — 消息队列多 Agent（进阶） | ⭐⭐ |
| Part 7 | 练习 C — Supervisor + HITL（挑战） | ⭐⭐⭐ |

### 学习目标
1. 理解 **AutoGen** 和 **CrewAI** 的核心抽象（Agent / Task / Team / Crew）
2. 掌握 **GraphFlow**（有向图调度）和 **SelectorGroupChat**（动态选择）两种协作模式
3. 实现包含「检索 → 生成 → 验证」的 **Agentic RAG** 流程
4. 完成三道练习，巩固多 Agent 系统设计能力

### 前置要求
- Python 3.10+
- `OPENROUTER_API_KEY` 环境变量（必需，前往 https://openrouter.ai 获取）
- `TAVILY_API_KEY` 环境变量（Part 4 / 练习 B 需要）

> **OpenRouter** 提供统一的 OpenAI 兼容接口，可路由到 Claude、GPT、Gemini 等多种模型。
> 模型列表：https://openrouter.ai/models

## 📖 讲义
### 课程资料

1. 引言（Introduction）
多 Agent 框架让我们把复杂任务拆成若干相互协作的“智能体”（Agent），每个 Agent 负责单一职责（比如检索、摘要、生成、校验、调度等）。AutoGen 与 CrewAI 是两类常见的多 Agent 框架/工具集：它们帮助你快速定义 Agent、设计通信协议、做任务分配与生命周期管理，并提供调试/监控能力。掌握这些框架，能显著降低构建生产级多 Agent 系统的门槛，让你更专注于业务逻辑与质量控制，而不是重复造调度/通信轮子。

2.1 Multi-Agent Frameworks 概述
目标：把复杂任务拆分成可并行、可重试、可扩展的子任务，由多个 Agent 协作完成。

关键能力：Agent 注册与发现、任务分配与调度、消息/事件通信、状态/trace 记录、故障隔离与重试、监控与审计。

常见模式：Pipeline（流水线）、Orchestrator（中心编排）、Blackboard（共享黑板）、Map-Reduce（并行/聚合）。

---

### 幻灯片

# Multi-Agent Frameworks
## AutoGen 与 CrewAI 实战指南

课程学习资料 · 面向零基础 AI Engineer 训练营

含 LangGraph 对比与选型 · 120 分钟 · 88 页

---

## 课程快照

- 主题：多 Agent 框架的核心思想、协作模式与生产化要点
- 框架：AutoGen、CrewAI，并引入 LangGraph 做横向比较
- 目标：看懂不同框架的抽象层级、协作方式与选型逻辑
- 输出：能把复杂任务拆成可协作的 Agents，并组织成稳定流程
- 重点：协作、调度、状态、监督、可观测性

> 讲师提示：先强调“框架不是目的，稳定协作才是目的”。

---

## 适合谁学

- 零基础 AI Engineer 训练营学员
- 会一点 Python 和 LLM 基础概念即可
- 不要求熟悉 AutoGen、CrewAI、LangGraph 真实 API
- 适合想把一个复杂任务拆成多个 Agent 的同学
- 适合准备做课程项目、Demo 或自动化工作流的同学

> 互动：请每位同学先想一个最想自动化的任务。

---

## 学习目标

- 理解 Multi-Agent Systems 的核心概念
- 学会用 AutoGen 设计原型级的协作流程
- 学会用 CrewAI 组织任务编排与执行
- 看懂 LangGraph 的 graph + state 心智模型
- 能比较三者并做基础选型
- 能在项目里加入 trace、quality gates 与 HITL

---

## 120 分钟路线图

| 时间 | 主题 | 目标 |
| --- | --- | --- |
| 0-15 分钟 | 为什么需要多 Agent | 建立直觉 |
| 15-35 分钟 | MAS 基础与协作模式 | 看懂架构 |
| 35-53 分钟 | AutoGen | 会做原型 |
| 53-71 分钟 | CrewAI | 会看编排 |
| 71-83 分钟 | LangGraph 与三框架对比 | 会做选型 |
| 83-101 分钟 | 协作、监督与生产化 | 学会工程化 |
| 101-113 分钟 | 实战场景 | 看懂数据流 |
| 113-120 分钟 | 练习与总结 | 收束到项目 |

---

## 一、为什么需要多 Agent

- 单个大模型很强，但复杂任务往往需要分工
- 多 Agent 可以把“一个大问题”拆成“多个小职责”
- 每个 Agent 专注一件事：检索、生成、校验、调度、总结
- 这会让系统更可控、更可测、更容易扩展
- 也更方便替换模型、工具和执行策略

> 讲师提示：把“协作”解释成工程分工，而不是拟人化聊天。

---

## 复杂任务的本质

- 任务通常跨越多个步骤
- 每一步有不同的成功标准
- 不同步骤适合不同模型、不同工具、不同速度
- 失败时不应该让整个系统一起倒掉
- 复杂系统需要可恢复、可追踪、可局部重试

例如：

- 先检索，再生成，再复核
- 先分类，再路由，再汇总
- 先计划，再执行，再验收

---

## Multi-Agent Systems 是什么

- 多个 Agent 组成一个协作系统
- 通过消息、任务或共享状态来协同
- 每个 Agent 都应有明确角色、职责和输入输出
- Orchestrator 负责组织流程，监督者负责质量
- 重点不是“有几个 Agent”，而是“边界是不是清楚”

关键目标：

- 并行
- 重试
- 隔离
- 追踪
- 扩展

---

## 常见架构模式

| 模式 | 适合场景 | 特点 |
| --- | --- | --- |
| Pipeline | 稳定流程 | 顺序清晰 |
| Orchestrator | 中心编排 | 控制力强 |
| Blackboard | 共享知识 | 协同灵活 |
| Map-Reduce | 并行聚合 | 扩展性好 |
| State Graph | 分支丰富流程 | 状态与路由显式 |

> 课堂互动：让学员把“日报生成”映射到一种模式。

---

## Agent 的生命周期

- 注册：Agent 被系统发现或装配
- 领取任务：从队列或调度器拿到工作
- 执行：调用模型、工具或外部系统
- 上报：返回结果、状态和 trace
- 终止或重试：根据策略继续或退出

好的生命周期管理能避免“任务做一半就失联”。

---

## 通信的基本选择

- 同步 RPC：适合即时反馈、单步决策
- 异步消息队列：适合扩展、缓冲、抗故障
- 共享状态：适合状态驱动的工作流
- 事件驱动：适合松耦合与回放
- 关键在于选对“协作载体”

选择原则：

- 低延迟优先用同步
- 高吞吐优先用异步
- 分支复杂优先显式状态

---

## 可观测性为什么重要

- Agent 系统很容易“看起来在跑，实际上在漂”
- trace 能帮助我们复盘每一步发生了什么
- 指标能告诉我们系统是不是变慢、变贵、变差
- 审计能告诉我们是谁访问了什么数据
- 没有观测，就很难判断问题出在模型、工具还是流程

常见观测对象：

- latency
- error rate
- queue length
- task success rate
- cost/token

---

## 监督与治理

- 质量门：在关键阶段插入 verifier
- 人工复核：高风险场景保留 HITL
- 权限控制：Agent 只拿到最小权限
- 数据保留：记录该记的，删除不该留的
- 预算控制：对高成本步骤设置上限

> 讲师提示：这里要强调“能自动化，不等于应该全自动化”。

> 📖 完整讲义已嵌入本节（原 `课程资料.md` / `Marp 幻灯片.md` 已删除）


In [ ]:
import subprocess, sys

packages = [
    'autogen-agentchat',
    'autogen-ext[openai]',
    'crewai',
    'crewai-tools',
    'langgraph',
    'langchain-openai',
    'langchain-community',
    'chromadb',
    'python-dotenv',
    'tavily-python',
]

print('📦 安装依赖中（可能需要 1-2 分钟）...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('❌ 安装失败：')
    print(result.stderr[-2000:])
else:
    print('✅ 安装完成')
    print()
    print('⚠️  重要：请点击菜单 Kernel → Restart Kernel，然后从 Part 1 重新运行。')
    print('   安装后必须重启内核，新模块才能生效。')


📦 安装依赖中（可能需要 1-2 分钟）...
❌ 安装失败：
/home/lancelot/src/AI-Engineer-Bootcamp/Lecture 27/.venv/bin/python: No module named pip



## Part 1 — 环境初始化

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# OpenRouter 配置
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

if not OPENROUTER_API_KEY:
    raise EnvironmentError(
        'OPENROUTER_API_KEY 未设置。\n'
        '请前往 https://openrouter.ai 注册并获取 API Key，'
        '然后在 .env 文件中添加：OPENROUTER_API_KEY=sk-or-v1-...'
    )

# 模型名称（OpenRouter 格式：provider/model-name）
# 完整列表：https://openrouter.ai/models
MODEL_SONNET = 'anthropic/claude-sonnet-4-5'   # 推理能力强，适合复杂任务
MODEL_HAIKU  = 'anthropic/claude-haiku-4-5'    # 响应速度快，适合轻量任务

print('✅ 环境变量加载成功')
print(f'   OPENROUTER_API_KEY : {OPENROUTER_API_KEY[:12]}...')
print(f'   TAVILY_API_KEY     : {TAVILY_API_KEY[:8] + "..." if TAVILY_API_KEY else "未设置（Part 4 需要）"}')
print(f'   Sonnet 模型        : {MODEL_SONNET}')
print(f'   Haiku  模型        : {MODEL_HAIKU}')

: 